In [1]:
# Libraries
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
# Master tables.
masterTableHuman = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/humanParalogy.txt", sep="\t")
masterTableMouse = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/mouseParalogy.txt", sep="\t")

# Add Expression Profile Information

In [3]:
gtexEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/gtexExpressionProfile.parquet")
emtabEP = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/emtabExpressionProfile.parquet")

# Merge the paralogue table with the expression profile data. Have to do this twice for the two genes.
masterTableHuman = masterTableHuman.merge(gtexEP.add_prefix("Gene 1 "), on="Gene stable ID", how="left")
masterTableHuman = masterTableHuman.merge(gtexEP.add_prefix("Gene 2 "), left_on="Human paralogue gene stable ID", right_on="Gene stable ID", how="left")

# Merge the paralogue table with the expression profile data. Have to do this twice for the two genes. 
masterTableMouse = masterTableMouse.merge(emtabEP.add_prefix("Gene 1 "), on="Gene stable ID", how="left")
masterTableMouse = masterTableMouse.merge(emtabEP.add_prefix("Gene 2 "), left_on="Mouse paralogue gene stable ID", right_on="Gene stable ID", how="left")

# Calculate Distances

In [4]:
def calcTEC(geneOne, geneTwo):
    # Turns the expression profiles into a binary vector that tells us whether a gene is expressed (TPM > 1) or not in a tissue. Turn the dataframe into a series.
    humanOrthoBinary = (geneOne[:-1] > 1)
    mouseOrthoBinary = (geneTwo[:-1] > 1)

    # Finds the number of tissues that are found in one species and not in the other.
    humanOnlyTissueNum = (humanOrthoBinary & ~mouseOrthoBinary).sum()
    mouseOnlyTissueNum = (mouseOrthoBinary & ~humanOrthoBinary).sum()

    # Using the binary vector, we can calculate the total number of tissues the gene is expressed in.
    humanTotalTissue = humanOrthoBinary.sum()
    mouseTotalTissue = mouseOrthoBinary.sum()

    # If any gene is not expressed in any tissue, the TEC formula will output an error. We handle this case by outputting NaN.
    if humanTotalTissue == 0 or mouseTotalTissue == 0:
        return np.nan
    else:
        return ((humanOnlyTissueNum / humanTotalTissue) + (mouseOnlyTissueNum / mouseTotalTissue)) / 2

In [5]:
# Grabs the parental and daughter copy expression profiles.
geneOneEPHuman = masterTableHuman.filter(like="Gene 1")
geneTwoEPHuman = masterTableHuman.filter(like="Gene 2")

# Calculates Euclidean distance.
masterTableHuman["EuclidDist"] = np.linalg.norm(geneOneEPHuman.select_dtypes(include="number").to_numpy() - geneTwoEPHuman.select_dtypes(include="number").to_numpy(), axis=1)

# Calculates Euclidean distance with Euclidean normalization applied. First calculates the Euclidean norm of each row (axis=1), then divides each "index" (row) by its respective Euclidean norm. 
masterTableHuman["EuclidDistNorm"] = np.linalg.norm(geneOneEPHuman.select_dtypes(include="number").div(np.linalg.norm(geneOneEPHuman.select_dtypes(include="number"), axis=1), axis=0).to_numpy() - geneTwoEPHuman.select_dtypes(include="number").div(np.linalg.norm(geneTwoEPHuman.select_dtypes(include="number"), axis=1), axis=0).to_numpy(), axis=1)

# Calculates Euclidean distance with log2 transformation applied.
masterTableHuman["EuclidDistLog"] = np.linalg.norm(np.log2(geneOneEPHuman.select_dtypes(include="number") + 1).to_numpy() - np.log2(geneTwoEPHuman.select_dtypes(include="number") + 1).to_numpy(), axis=1)                                                                                                                                                                                                                                                                                                                                                                                                                                                                

# Calculates Pearson distance. The "corrwith" function requires dataframes to have the same column name. That's why I had to rename geneTwoEPHuman's columns to geneOneEPHuman'
masterTableHuman["PearDist"] = 1 - geneOneEPHuman.select_dtypes(include="number").corrwith(geneTwoEPHuman.select_dtypes(include="number").rename(columns=dict(zip(geneTwoEPHuman.columns, geneOneEPHuman.columns))), axis=1, method="pearson").to_numpy()

# Iterates through each row in both dataframes and calculates their TEC score.
masterTableHuman["TEC"] = [calcTEC(parentCopy, geneTwoCopy) for parentCopy, geneTwoCopy in zip(geneOneEPHuman.select_dtypes(include="number").to_numpy(), geneTwoEPHuman.select_dtypes(include="number").to_numpy())]

In [6]:
# Grabs the geneOne and geneTwo copy expression profiles.
geneOneEPMouse = masterTableMouse.filter(like="Gene 1")
geneTwoEPMouse = masterTableMouse.filter(like="Gene 2")

# Calculates Euclidean distance.
masterTableMouse["EuclidDist"] = np.linalg.norm(geneOneEPMouse.select_dtypes(include="number").to_numpy() - geneTwoEPMouse.select_dtypes(include="number").to_numpy(), axis=1)

# Calculates Euclidean distance with Euclidean normalization applied. First calculates the Euclidean norm of each row (axis=1), then divides each "index" (row) by its respective Euclidean norm. 
masterTableMouse["EuclidDistNorm"] = np.linalg.norm(geneOneEPMouse.select_dtypes(include="number").div(np.linalg.norm(geneOneEPMouse.select_dtypes(include="number"), axis=1), axis=0).to_numpy() - geneTwoEPMouse.select_dtypes(include="number").div(np.linalg.norm(geneTwoEPMouse.select_dtypes(include="number"), axis=1), axis=0).to_numpy(), axis=1)

# Calculates Euclidean distance with log2 transformation applied.
masterTableMouse["EuclidDistLog"] = np.linalg.norm(np.log2(geneOneEPMouse.select_dtypes(include="number") + 1).to_numpy() - np.log2(geneTwoEPMouse.select_dtypes(include="number") + 1).to_numpy(), axis=1)                                                                                                                                                                                                                                                                                                                                                                                                                                                                

# Calculates Pearson distance. The "corrwith" function requires dataframes to have the same column name. That's why I had to rename geneTwoEPMouse's columns to geneOneEPMouse'
masterTableMouse["PearDist"] = 1 - geneOneEPMouse.select_dtypes(include="number").corrwith(geneTwoEPMouse.select_dtypes(include="number").rename(columns=dict(zip(geneTwoEPMouse.columns, geneOneEPMouse.columns))), axis=1, method="pearson").to_numpy()

# Iterates through each row in both dataframes and calculates their TEC score.
masterTableMouse["TEC"] = [calcTEC(parentCopy, geneTwoCopy) for parentCopy, geneTwoCopy in zip(geneOneEPMouse.select_dtypes(include="number").to_numpy(), geneTwoEPMouse.select_dtypes(include="number").to_numpy())]

# Number of Duplicates

In [7]:
humanPairs = masterTableHuman.iloc[:, 0:2]
mousePairs = masterTableMouse.iloc[:, 0:2]

In [8]:
masterTableHuman = masterTableHuman.merge(humanPairs.groupby("Gene stable ID").count(), left_on="Gene stable ID", right_index=True, how="left").rename(columns={"Human paralogue gene stable ID_y": "Number of Duplicates"})
masterTableMouse = masterTableMouse.merge(mousePairs.groupby("Gene stable ID").count(), left_on="Gene stable ID", right_index=True, how="left").rename(columns={"Mouse paralogue gene stable ID_y": "Number of Duplicates"})

# Exons

In [9]:
humanExons = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/humanExons.txt", sep="\t")
mouseExons = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/mouseExons.txt", sep="\t")

humanExonsGrouped = humanExons.groupby("Gene stable ID")["Exon stable ID"].agg(lambda x: ", ".join(x.unique()))
mouseExonsGrouped = mouseExons.groupby("Gene stable ID")["Exon stable ID"].agg(lambda x: ", ".join(x.unique()))

In [10]:
masterTableHuman = masterTableHuman.merge(humanExonsGrouped, on="Gene stable ID", how="left")
masterTableMouse = masterTableMouse.merge(mouseExonsGrouped, on="Gene stable ID", how="left")

In [11]:
masterTableHuman

,Gene stable ID,Human paralogue gene stable ID_x,Human paralogue homology type,Paralogue last common ancestor with Human,Gene 1 Brain,Gene 1 Colon,Gene 1 Esophagus,Gene 1 Heart,Gene 1 Kidney,Gene 1 Liver,...,Gene 2 Pancreas,Gene 2 Stomach,Gene 2 Gene type,EuclidDist,EuclidDistNorm,EuclidDistLog,PearDist,TEC,Number of Duplicates,Exon stable ID
0,ENSG00000210049,NaN,NaN,NaN,17.516844,1.400129,0.999184,3.366039,4.574239,0.597851,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,ENSE00001544501
1,ENSG00000211459,NaN,NaN,NaN,23502.966797,3544.858643,3435.218750,7646.939941,10045.823242,3707.004150,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,ENSE00001544499
2,ENSG00000210077,NaN,NaN,NaN,18.865494,1.571674,1.380163,3.266439,3.701754,0.723936,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,ENSE00001544498
3,ENSG00000210082,NaN,NaN,NaN,72589.617188,18863.207031,16807.601562,39093.738281,51649.222656,20325.992188,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,ENSE00001544497
4,ENSG00000209082,NaN,NaN,NaN,25.209278,6.738225,4.899840,15.493462,11.392898,4.587103,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,ENSE00002006242
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3625954,ENSG00000157873,ENSG00000243509,other_paralog,Chordata,12.655311,120.186638,69.148285,20.617067,101.015106,54.591118,...,1.235028,4.021246,protein_coding,189.903580,0.500351,10.446088,0.463805,0.142857,21,"ENSE00001759361, ENSE00001576845, ENSE00003529..."
3625955,ENSG00000157873,ENSG00000026103,other_paralog,Chordata,12.655311,120.186638,69.148285,20.617067,101.015106,54.591118,...,0.426707,1.094653,protein_coding,196.172974,0.323165,12.601530,0.198451,0.214286,21,"ENSE00001759361, ENSE00001576845, ENSE00003529..."
3625956,ENSG00000157873,ENSG00000120949,other_paralog,Chordata,12.655311,120.186638,69.148285,20.617067,101.015106,54.591118,...,0.078418,1.143263,protein_coding,198.702774,0.713202,14.173954,0.937936,0.428571,21,"ENSE00001759361, ENSE00001576845, ENSE00003529..."
3625957,ENSG00000132676,NaN,NaN,NaN,23.821829,46.711121,42.119331,28.719112,30.156294,29.079199,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,"ENSE00001936122, ENSE00001845528, ENSE00001811..."


In [12]:
masterTableHuman["Number of Exons"] = masterTableHuman["Exon stable ID"].str.split(", ").str.len()
masterTableMouse["Number of Exons"] = masterTableMouse["Exon stable ID"].str.split(", ").str.len()

# Identifying In/Out-Paralogs

In [13]:
masterTableMouse["Paralogue last common ancestor with Mouse"].unique()

<ArrowStringArray>
[                                      nan,
                               'Bilateria',
 'Mus musculus reference (CL57BL6) strain',
                                     'Mus',
                                 'Murinae',
                                'Muroidea',
                                'Rodentia',
                                'Eutheria',
                           'Gnathostomata',
                              'Vertebrata',
                                'Mammalia',
                           'Boreoeutheria',
                            'Euteleostomi',
                                'Chordata',
                                 'Amniota',
                            'Opisthokonta',
                                  'Theria',
                           'Sarcopterygii',
                               'Tetrapoda',
                               'Myomorpha',
                        'Euarchontoglires',
                                  'Glires']
Length: 22, d

In [14]:
humanSpecific = ["Primates", "Haplorrhini", "Simiiformes", "Catarrhini", "Hominidae", "Homininae", "Homo sapiens"]
mouseSpecific = ["Glires", "Rodentia", "Myomorpha", "Muroidea", "Murinae", "Mus", "Mus musculus reference (CL57BL6) strain"]

humanParalogyConditions = [
    masterTableHuman["Paralogue last common ancestor with Human"].isin(humanSpecific),
    (~masterTableHuman["Paralogue last common ancestor with Human"].isin(humanSpecific)) & (~masterTableHuman["Paralogue last common ancestor with Human"].isna()),
]
mouseParalogyConditions = [
    masterTableMouse["Paralogue last common ancestor with Mouse"].isin(mouseSpecific),
    (~masterTableMouse["Paralogue last common ancestor with Mouse"].isin(mouseSpecific)) & (~masterTableMouse["Paralogue last common ancestor with Mouse"].isna()),
]

paralogyChoices = [
    "in-paralog",
    "out-paralog"
]

masterTableHuman["Paralogy Type"] = np.select(humanParalogyConditions, paralogyChoices, default="NA")
masterTableMouse["Paralogy Type"] = np.select(mouseParalogyConditions, paralogyChoices, default="NA")

# Create Master Tables

In [15]:
newColOrderHuman = list(masterTableHuman.columns[0:3]) + list(masterTableHuman.columns[-1:]) + list(masterTableHuman.columns[3:4]) + list(masterTableHuman.columns[-8:-3]) + list(masterTableHuman.columns[-3:-2]) + list(masterTableHuman.columns[-2:-3]) + list(masterTableHuman.columns[12:13]) + list(masterTableHuman.columns[21:22]) + list(masterTableHuman.columns[4:12]) + list(masterTableHuman.columns[13:21]) + list(masterTableHuman.columns[-2:-1])
newColOrderMouse = list(masterTableMouse.columns[0:3]) + list(masterTableMouse.columns[-1:]) + list(masterTableMouse.columns[3:4]) + list(masterTableMouse.columns[-8:-3]) + list(masterTableMouse.columns[-3:-2]) + list(masterTableMouse.columns[-2:-3]) + list(masterTableMouse.columns[12:13]) + list(masterTableMouse.columns[21:22]) + list(masterTableMouse.columns[4:12]) + list(masterTableMouse.columns[13:21]) + list(masterTableMouse.columns[-2:-1])

masterTableHuman = masterTableHuman.loc[:, newColOrderHuman].rename(columns={"Human paralogue gene stable ID_x": "Human paralogue gene stable ID"})
masterTableMouse = masterTableMouse.loc[:, newColOrderMouse].rename(columns={"Mouse paralogue gene stable ID_x": "Mouse paralogue gene stable ID"})

In [16]:
masterTableHuman.to_csv("/Users/andrewhsu/Projects/McNair/data/humanParalogMasterTable.csv", index=False)
masterTableHuman.to_parquet("/Users/andrewhsu/Projects/McNair/data/humanParalogMasterTable.parquet", index=False)

masterTableMouse.to_csv("/Users/andrewhsu/Projects/McNair/data/mouseParalogMasterTable.csv", index=False)
masterTableMouse.to_parquet("/Users/andrewhsu/Projects/McNair/data/mouseParalogMasterTable.parquet", index=False)

In [17]:
display(masterTableHuman)
display(masterTableMouse)

,Gene stable ID,Human paralogue gene stable ID,Human paralogue homology type,Paralogy Type,Paralogue last common ancestor with Human,EuclidDistNorm,EuclidDistLog,PearDist,TEC,Number of Duplicates,...,Gene 1 Stomach,Gene 2 Brain,Gene 2 Colon,Gene 2 Esophagus,Gene 2 Heart,Gene 2 Kidney,Gene 2 Liver,Gene 2 Pancreas,Gene 2 Stomach,Number of Exons
0,ENSG00000210049,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,0,...,1.578150,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,ENSG00000211459,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,0,...,4341.306641,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,ENSG00000210077,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,0,...,1.512026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,ENSG00000210082,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,0,...,19898.167969,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,ENSG00000209082,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,0,...,6.185790,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3625954,ENSG00000157873,ENSG00000243509,other_paralog,out-paralog,Chordata,0.500351,10.446088,0.463805,0.142857,21,...,82.190369,5.599104,6.123305,5.789531,0.963287,4.668557,0.847602,1.235028,4.021246,79
3625955,ENSG00000157873,ENSG00000026103,other_paralog,out-paralog,Chordata,0.323165,12.601530,0.198451,0.214286,21,...,82.190369,0.621104,2.578365,2.349528,0.991792,1.590631,1.228026,0.426707,1.094653,79
3625956,ENSG00000157873,ENSG00000120949,other_paralog,out-paralog,Chordata,0.713202,14.173954,0.937936,0.428571,21,...,82.190369,0.856435,0.709645,0.858565,1.057785,0.412732,0.098749,0.078418,1.143263,79
3625957,ENSG00000132676,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,0,...,41.685940,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,131


,Gene stable ID,Mouse paralogue gene stable ID,Mouse paralogue homology type,Paralogy Type,Paralogue last common ancestor with Mouse,EuclidDistNorm,EuclidDistLog,PearDist,TEC,Number of Duplicates,...,Gene 1 Stomach,Gene 2 Brain,Gene 2 Colon,Gene 2 Esophagus,Gene 2 Heart,Gene 2 Kidney,Gene 2 Liver,Gene 2 Pancreas,Gene 2 Stomach,Number of Exons
0,ENSMUSG00000064336,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,0,...,2.508600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,ENSMUSG00000064337,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,0,...,1466.738960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,ENSMUSG00000064338,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,0,...,1.402663,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,ENSMUSG00000064339,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,0,...,1335.432013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,ENSMUSG00000064340,NaN,NaN,NA,NaN,NaN,NaN,NaN,NaN,0,...,150.967196,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2426744,ENSMUSG00000026833,ENSMUSG00000022026,other_paralog,out-paralog,Bilateria,1.313149,10.015711,1.092443,0.250000,10,...,6.506703,2.131084,0.156930,0.167683,1.709669,15.643258,0.219291,0.411394,6.718520,14
2426745,ENSMUSG00000026833,ENSMUSG00000027848,other_paralog,out-paralog,Bilateria,1.040515,6.428900,0.714240,0.083333,10,...,6.506703,10.394712,8.815712,11.619676,5.965840,13.253190,0.866962,0.266533,6.503741,14
2426746,ENSMUSG00000026833,ENSMUSG00000038463,other_paralog,out-paralog,Bilateria,1.274453,7.389735,1.093196,0.083333,10,...,6.506703,6.205680,17.968269,31.864427,6.978360,1.881622,0.266059,0.678200,7.194103,14
2426747,ENSMUSG00000026833,ENSMUSG00000046167,other_paralog,out-paralog,Bilateria,0.927514,11.691460,0.525570,0.416667,10,...,6.506703,0.761520,0.007213,1.142975,0.031373,0.006148,0.031491,0.000000,0.005854,14


# Statistical Testing

In [18]:
statisticsDFHuman = pd.DataFrame({
    "Category": ["out-paralog", "in-paralog"],
    "n": 0.0,
    "Median": 0.0,
    "Mean": 0.0, 
    "Rho": 0.0,
    "Corr PValue" : 0.0
})

statisticsDFMouse = pd.DataFrame({
    "Category": ["out-paralog", "in-paralog"],
    "n": 0.0,
    "Median": 0.0,
    "Mean": 0.0, 
    "Rho": 0.0,
    "Corr PValue" : 0.0
})

In [19]:
sigHumanDF = pd.DataFrame({
    "Group 1": ["out-paralog"],
    "Group 2": ["in-paralog"],
    "PValue": 0.0
})

sigMouseDF = pd.DataFrame({
    "Group 1": ["out-paralog"],
    "Group 2": ["in-paralog"],
    "PValue": 0.0
})


In [20]:
dfListHuman = []
dfListMouse = []
for distMetric in masterTableHuman.columns[5:10]:
    nHumanArr = []
    nMouseArr = []
    medianHumanArr = []
    medianMouseArr = []
    meanHumanArr = []
    meanMouseArr = []
    corrHumanArr = []
    corrMouseArr = []
    pValueHumanArr = []
    pValueMouseArr = []

    for category in statisticsDFHuman["Category"]:
        filteredHumanDF = masterTableHuman[masterTableHuman["Paralogy Type"] == category][distMetric].dropna()
        filteredMouseDF = masterTableMouse[masterTableMouse["Paralogy Type"] == category][distMetric].dropna()

        nHumanArr.append(filteredHumanDF.shape[0])
        nMouseArr.append(filteredMouseDF.shape[0])
        medianHumanArr.append(filteredHumanDF.median())
        medianMouseArr.append(filteredMouseDF.median())
        meanHumanArr.append(filteredHumanDF.mean())
        meanMouseArr.append(filteredMouseDF.mean())

        corrHuman = stats.spearmanr(masterTableHuman[(masterTableHuman["Paralogy Type"] == category) & (~masterTableHuman[distMetric].isna())][distMetric], masterTableHuman[(masterTableHuman["Paralogy Type"] == category) & (~masterTableHuman[distMetric].isna())]["Number of Duplicates"])
        corrMouse = stats.spearmanr(masterTableMouse[(masterTableMouse["Paralogy Type"] == category) & (~masterTableMouse[distMetric].isna())][distMetric], masterTableMouse[(masterTableMouse["Paralogy Type"] == category) & (~masterTableMouse[distMetric].isna())]["Number of Duplicates"])
        corrHumanArr.append(corrHuman[0])
        corrMouseArr.append(corrMouse[0])
        pValueHumanArr.append(corrHuman[1])
        pValueMouseArr.append(corrMouse[1])

    statisticsDFHuman["n"] = nHumanArr
    statisticsDFHuman["Median"] = medianHumanArr
    statisticsDFHuman["Mean"] = meanHumanArr
    statisticsDFHuman["Rho"] = corrHumanArr
    statisticsDFHuman["Corr PValue"] = pValueHumanArr
    statisticsDFMouse["n"] = nMouseArr
    statisticsDFMouse["Median"] = medianMouseArr
    statisticsDFMouse["Mean"] = meanMouseArr
    statisticsDFMouse["Rho"] = corrMouseArr
    statisticsDFMouse["Corr PValue"] = pValueMouseArr

    dfListHuman.append(statisticsDFHuman.copy())
    dfListMouse.append(statisticsDFMouse.copy())

In [21]:
dfListHuman[0]

,Category,n,Median,Mean,Rho,Corr PValue
0,out-paralog,3228272,0.810108,0.810108,0.178815,0.000000e+00
1,in-paralog,6890,0.431129,0.518971,-0.065326,5.721087e-08


In [22]:
dfListHumanSig = []
dfListMouseSig = []
for distMetric in masterTableHuman.columns[5:10]:
    pValueHumanArr = []
    pValueMouseArr = []
    for idx in range(len(sigHumanDF["Group 1"])):
        filteredHumanDF1 = masterTableHuman[masterTableHuman["Paralogy Type"] == sigHumanDF["Group 1"][idx]][distMetric].dropna()
        filteredHumanDF2 = masterTableHuman[masterTableHuman["Paralogy Type"] == sigHumanDF["Group 2"][idx]][distMetric].dropna()
        filteredMouseDF1 = masterTableMouse[masterTableMouse["Paralogy Type"] == sigMouseDF["Group 1"][idx]][distMetric].dropna()
        filteredMouseDF2 = masterTableMouse[masterTableMouse["Paralogy Type"] == sigMouseDF["Group 2"][idx]][distMetric].dropna()

        humanSig = stats.mannwhitneyu(filteredHumanDF1, filteredHumanDF2)[1]
        mouseSig = stats.mannwhitneyu(filteredMouseDF1, filteredMouseDF2)[1]

        pValueHumanArr.append(humanSig)
        pValueMouseArr.append(mouseSig)
    sigHumanDF["PValue"] = pValueHumanArr
    sigMouseDF["PValue"] = pValueMouseArr

    dfListHumanSig.append(sigHumanDF.copy())
    dfListMouseSig.append(sigMouseDF.copy())
        

In [23]:
with pd.ExcelWriter("/Users/andrewhsu/Projects/McNair/data/humanParalogStatistics.xlsx") as w:
    for idx, distMetric in enumerate(masterTableHuman.columns[5:10]):
        dfListHuman[idx].to_excel(w, sheet_name=distMetric, index=False)
        dfListHumanSig[idx].to_excel(w, sheet_name=distMetric, index=False, startrow=0, startcol=7)
    
with pd.ExcelWriter("/Users/andrewhsu/Projects/McNair/data/mouseParalogStatistics.xlsx") as w:
    for idx, distMetric in enumerate(masterTableMouse.columns[5:10]):
        dfListMouse[idx].to_excel(w, sheet_name=distMetric, index=False)
        dfListMouseSig[idx].to_excel(w, sheet_name=distMetric, index=False, startrow=0, startcol=7)